In [ ]:
import json
import networkx as nx
from networkx.readwrite import json_graph
import glob

G = nx.DiGraph()

# 1. Load the seeded JSON programs
program_files = glob.glob("data/programs/*.json")

for file in program_files:
    with open(file, 'r') as f:
        prog = json.load(f)

        # Add Program Node
        G.add_node(prog["program_id"], type="Program", name=prog["name"],
                   speed=prog["speed_to_fund"], amount=prog["max_amount"])

        # Add Org Node & Edge
        G.add_node(prog["org_id"], type="Org", name=prog["org_name"])
        G.add_edge(prog["program_id"], prog["org_id"], relationship="OFFERED_BY")

        # Add Criteria Nodes & Edges
        # Corrected from 'criteria' to 'eligibility_criteria' based on the schema
        for crit in prog.get("eligibility_criteria", []):
            G.add_node(crit["id"], type="Criterion", description=crit["description"], field=crit.get("field"), operator=crit.get("operator"), value=crit.get("value"))
            G.add_edge(prog["program_id"], crit["id"], relationship="REQUIRES", condition_type=crit["type"])

        # Add Document Nodes & Edges
        for doc in prog.get("documents_needed", []):
            G.add_node(doc, type="DocumentType")
            G.add_edge(prog["program_id"], doc, relationship="REQUIRES_DOC")

# Export to node-link format for the backend
data = json_graph.node_link_data(G)
with open("data/graph_snapshot.json", "w") as f:
    json.dump(data, f, indent=4)

print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")